In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from collections import deque
import heapq
import unittest
from scipy import stats
import copy as cp
from time import time

---

## Game Theory - Playing "intelligent" Tic-Tac-Toe

<img src="https://www.cookieshq.co.uk/images/2016/06/01/tic-tac-toe.png" width="150"/>



### (1a)   Defining the Tic-Tac-Toe class structure

Fill in this class structure for Tic-Tac-Toe using what we did during class as a guide.
* `moves` is a list of tuples to represent which moves are available. Recall that we are using matrix notation for this, where the upper-left corner of the board, for example, is represented at (1,1).
* `result(self, move, state)` returns a ***hypothetical*** resulting `State` object if `move` is made when the game is in the current `state`
* `compute_utility(self, move, state)` calculates the utility of `state` that would result if `move` is made when the game is in the current `state`. This is where you want to check to see if anyone has gotten `nwin` in a row
* `game_over(self, state)` - this wasn't a method, but it should be - it's a piece of code we need to execute repeatedly and giving it a name makes clear what task the piece of code performs. Returns `True` if the game in the given `state` has reached a terminal state, and `False` otherwise.
* `utility(self, state, player)` also wasn't a method earlier, but also should be.  Returns the utility of the current state if the player is X and $-1 \cdot$ utility if the player is O.
* `display(self)` is a method to display the current game `state`, You get it for free! because this would be super frustrating without it.
* `play_game(self, player1, player2)` returns an integer that is the utility of the outcome of the game (+1 if X wins, 0 if draw, -1 if O wins). `player1` and `player2` are functional arguments that we will deal with in parts **1b** and **1d**.

Some notes:
* Assume X always goes first.
* Do **not** hard-code for 3x3 boards.
* You may add attributes and methods to these classes as needed for this problem.

In [2]:
class State:
    def __init__(self, moves):
        self.to_move = 'X'
        self.utility = 0
        self.board = {}
        self.moves = cp.copy(moves)

        
class TicTacToe:
    
    def __init__(self, nrow=3, ncol=3, nwin=3, nexp=0):
        self.nrow = nrow
        self.ncol = ncol
        self.nwin = nwin
        moves = [(row, col) for row in range(1, nrow + 1) for col in range(1, ncol + 1)]
        self.state = State(moves)
        self.nexp = nexp

    def result(self, move, state):
        '''
        What is the hypothetical result of move `move` in state `state` ?
        move  = (row, col) tuple where player will put their mark (X or O)
        state = a `State` object, to represent whose turn it is and form
                the basis for generating a **hypothetical** updated state
                that will result from making the given `move`
        '''
        
        # Don't do anything if the move isn't a legal one
        if move not in state.moves:
            print("Move not legal")
            return state
        # Return a copy of the updated state:
        #   compute utility, update the board, remove the move, update whose turn
        new_state = cp.deepcopy(state)
        new_state.utility = self.compute_utility(move, state)
        new_state.board[move] = state.to_move
        new_state.moves.remove(move)
        new_state.to_move = ('O' if state.to_move == 'X' else 'X')
        
        return new_state
        
    def compute_utility(self, move, state):
        '''
        What is the utility of making move `move` in state `state`?
        If 'X' wins with this move, return 1;
        if 'O' wins return -1;
        else return 0.
        '''        
        
        row, col = move
        player = state.to_move
        
        # create a hypothetical copy of the board, with 'move' executed
        board = cp.deepcopy(state.board)
        board[move] = player

        # what are all the ways 'player' could with with 'move'?
        
        # check for row-wise win
        in_a_row = 0
        for c in range(1,self.ncol+1):
            in_a_row += board.get((row,c))==player

        # check for column-wise win
        in_a_col = 0
        for r in range(1,self.nrow+1):
            in_a_col += board.get((r,col))==player

        # check for NW->SE diagonal win
        in_a_diag1 = 0
        for r in range(row,0,-1):
            in_a_diag1 += board.get((r,col-(row-r)))==player
        for r in range(row+1,self.nrow+1):
            in_a_diag1 += board.get((r,col-(row-r)))==player

        # check for SW->NE diagonal win
        in_a_diag2 = 0
        for r in range(row,0,-1):
            in_a_diag2 += board.get((r,col+(row-r)))==player
        for r in range(row+1,self.nrow+1):
            in_a_diag2 += board.get((r,col+(row-r)))==player
        
        if self.nwin in [in_a_row, in_a_col, in_a_diag1, in_a_diag2]:
            return 1 if player=='X' else -1
        else:
            return 0 
        
    def game_over(self, state):
        '''game is over if someone has won (utility!=0) or there
        are no more moves left'''

        # your code goes here
        return state.utility!=0 or len(state.moves)==0    

    
    def utility(self, state, player):
        '''Return the value to player; 1 for win, -1 for loss, 0 otherwise.'''

        return state.utility if player=='X' else -state.utility
       
    def display(self):
        for row in range(1, self.nrow+1):
            for col in range(1, self.ncol+1):
                print(self.state.board.get((row, col), '.'), end=' ')
            print()
        print()
        
    def play_game(self, player1, player2, display=True):
        '''Play a game of tic-tac-toe!'''

        turn_limit = self.nrow*self.ncol  # limit in case of buggy code
        turn = 0
        while turn <= turn_limit:
            for player in [player1, player2]:
                turn += 1
                move = player(self)
                self.state = self.result(move, self.state)
                if self.game_over(self.state):
                    if display == True: self.display()
                    return self.state.utility
    
    def play_game_with_me(self, player1, player2, display=True):
        '''Play a game of tic-tac-toe!'''

        turn_limit = self.nrow*self.ncol  # limit in case of buggy code
        turn = 0
        while turn <= turn_limit:
            for player in [player1, player2]:
                turn += 1
                
                if(player == player1):
                    move = player(self)
                    print(move)
                    self.state = self.result(move, self.state)
                else:
                    row = int(input("Row:"))
                    column = int(input("Column:"))
                    move = (row, column)
                    self.state = self.result(move, self.state)
                    self.display()
                if self.game_over(self.state):
                    if display == True: self.display()
                    return self.state.utility

### (1b) Define a random player

Define a function `random_player` that takes a single argument of the `TicTacToe` class and returns a random move out of the available legal moves in the `state` of the `TicTacToe` game.

In your code for the `play_game` method above, make sure that `random_player` could be a viable input for the `player1` and/or `player2` arguments.

In [3]:
def random_player(game):
    '''A player that chooses a legal move at random out of all
    available legal moves in Tic-Tac-Toe state argument'''

    possible_actions = game.state.moves
    return possible_actions[np.random.randint(low=0, high=len(possible_actions))]

We know from experience and/or because I'm telling you right now that if two `random_player`s play many games of Tic-Tac-Toe against one another, whoever goes first will win about 58% of the time.  Verify that this is the case by playing at least 1,000 games between two random players. Report the proportion of the games that the first player has won.

**"Unit tests":** If you are wondering how close is close enough to 58%, when I ran some tests, the min-max range of win percentage by the first player was 54-63%. Getting results around this range should be good.

In [4]:
# 1000 games between two random players
numGames = 1000
wins = 0
draws = 0
losses = 0
for k in range(numGames):
    ttt = TicTacToe(3,3,3)
    outcome = ttt.play_game(random_player, random_player, display = False)
    if outcome==1:
        wins += 1
    elif outcome==-1:
        losses += 1
    else:
        draws += 1

print('random_player vs random_player:')
print('First-player winning percentage = {}'.format(100*(wins/numGames)), "%")
print('First-player losing percentage = {}'.format(100*(losses/numGames)), "%")
print('Draw percentage = {}'.format(100*(draws/numGames)), "%")

random_player vs random_player:
First-player winning percentage = 60.5 %
First-player losing percentage = 27.6 %
Draw percentage = 11.899999999999999 %


### (1c) What about playing randomly on different-sized boards?

What does the long-term win percentage appear to be for the first player in a 4x4 Tic-Tac-Toe tournament, where 4 marks must be connected for a win?  Support your answer using a simulation and printed output, similar to **1b**.

**Also:** The win percentage should have changed substantially. Did the decrease in wins turn into more losses for the first player or more draws? Write a few sentences explaining the behavior you observed.  *Hint: think about how the size of the state space has changed.*

In [5]:
# 1000 games between two random players
numGames = 1000
wins = 0
draws = 0
losses = 0
for k in range(numGames):
    ttt = TicTacToe(4,4,4)
    outcome = ttt.play_game(random_player, random_player, display = False)
    if outcome==1:
        wins += 1
    elif outcome==-1:
        losses += 1
    else:
        draws += 1
        
print('random_player vs random_player on a 4x4 board:')
print('First-player winning percentage = {}'.format(100*(wins/numGames)), "%")
print('First-player losing percentage = {}'.format(100*(losses/numGames)), "%")
print('Draw percentage = {}'.format(100*(draws/numGames)), "%")

random_player vs random_player on a 4x4 board:
First-player winning percentage = 32.4 %
First-player losing percentage = 24.9 %
Draw percentage = 42.699999999999996 %


The decrease in wins for the first player seemed to be from an increased ammount of draws. This makes sense becuase the state space increased to 4x4 which is even in comparison to the 3x3 state space which is odd. In the even state space, both players get an an equal ammount of moves. In the odd state space, the first player gets an extra move which means they had an extra chance to win. It makes sence that in the even 4x4 state space there are more draws becuase both players will fill the same ammount of spaces. However, in both cases, the first player has a better chance at picking a "good" move first which could be why the first player's win percentage is still higher. 

### (1d) Define an alpha-beta player

Alright. Let's finally get serious about our Tic-Tac-Toe game.  No more fooling around!

Craft a function called `alphabeta_player` that takes a single argument of a `TicTacToe` class object and returns the minimax move in the `state` of the `TicTacToe` game. As the name implies, this player should be implementing alpha-beta pruning as described in the textbook and lecture.

Note that your alpha-beta search for the minimax move should include function definitions for `max_value` and `min_value` (see the aggressively realistic pseudocode from the lecture slides).

In your code for the `play_game` method above, make sure that `alphabeta_player` could be a viable input for the `player1` and/or `player2` arguments.

In [6]:
def alphabeta_player(ttt, prune = True, expandedTest = False):
    global expanded # made a global varible for expanded nodes because it makes things simpler
    
    # created a function maxFirstIter to isolate the first iteration of max_value function.
    # all that matters is the best path for the first iteration.
    # this will find the best value from all its possible moves and return the move that led there.
    # the bottom two functions just return numbers according to utility values.
    def maxFirstIter(state, alpha, beta, XorO, prune):
        value = -np.inf
        # this is the list of moves the algorithm will choose from
        for move in state.moves: 
            # calls min_value which will start the tradeoff between the min and max functions
            update = min_value(ttt.result(move, state), alpha, beta, XorO, prune) 
            # increment expanded everytime a new state is explored
            global expanded ; expanded += 1 
            if update > value:
                value = update
                bestMove = move
        return bestMove
    
    def max_value(state, alpha, beta, XorO, prune):
        # differentiates between X or O players because O is trying to get low
        # numbers and X wants high numbers, so the utility needs to inverse accordingly
        if ttt.game_over(state) and XorO == 'X': return state.utility 
        elif ttt.game_over(state): return -state.utility              
        value = -np.inf
        for move in state.moves:
            value = max(value, min_value(ttt.result(move, state), alpha, beta, XorO, prune))
            global expanded ; expanded += 1
            if prune:
                if value >= beta: return value
                alpha = max(value, alpha)
        return value
    
    def min_value(state, alpha, beta, XorO, prune):
        if ttt.game_over(state) and XorO == 'X': return state.utility
        elif ttt.game_over(state): return -state.utility
        value = np.inf
        for move in state.moves:
            value = min(value, max_value(ttt.result(move, state), alpha, beta, XorO, prune))
            global expanded ; expanded += 1 
            if prune:
                if value <= alpha: return value
                beta = min(value, beta)
        return value
    
    alpha = -np.inf
    beta = np.inf
    # varible to differentiate between X or O players
    XorO = ttt.state.to_move 
    # global varible expanded set to 0
    expanded = 0 
    bestMove = maxFirstIter(ttt.state, alpha, beta, XorO, prune)
    if expandedTest: return expanded
    else: return bestMove

Verify that your alpha-beta player code is working appropriately through the following tests, using a standard 3x3 Tic-Tac-Toe board. Run **10 games for each test**, and track the number of wins, draws and losses. Report these results for each case.

1. An alpha-beta player who plays first should never lose to a random player who plays second.
2. Two alpha-beta players should always draw.

**Nota bene:** Test your code with fewer games between the players to start, because the alpha-beta player will require substantially more compute time than the random player.  This is why I only ask for 10 games, which still might take a minute or two. You are welcome to run more than 10 tests if you'd like. 

In [7]:
numGames = 10
wins = 0
draws = 0
losses = 0
for k in range(numGames):
    ttt = TicTacToe(3,3,3)
    outcome = ttt.play_game(alphabeta_player, random_player, display = False)
    if outcome==1:
        wins += 1
    elif outcome==-1:
        losses += 1
    else:
        draws += 1
        
print('alphabeta_player as first player vs random_player:')
print('First-player winning percentage = {}'.format(100*(wins/numGames)), "%")
print('First-player losing percentage = {}'.format(100*(losses/numGames)), "%")
print('Draw percentage = {}'.format(100*(draws/numGames)), "%")
print()

numGames = 10
wins = 0
draws = 0
losses = 0
for k in range(numGames):
    ttt = TicTacToe(3,3,3)
    outcome = ttt.play_game(alphabeta_player, alphabeta_player, display = False)
    if outcome==1:
        wins += 1
    elif outcome==-1:
        losses += 1
    else:
        draws += 1
        
print('alphabeta_player vs alphabeta_player:')
print('First-player winning percentage = {}'.format(100*(wins/numGames)), "%")
print('First-player losing percentage = {}'.format(100*(losses/numGames)), "%")
print('Draw percentage = {}'.format(100*(draws/numGames)), "%")

alphabeta_player as first player vs random_player:
First-player winning percentage = 100.0 %
First-player losing percentage = 0.0 %
Draw percentage = 0.0 %

alphabeta_player vs alphabeta_player:
First-player winning percentage = 0.0 %
First-player losing percentage = 0.0 %
Draw percentage = 100.0 %


### (1e) What has pruning ever done for us?

Calculate the number of number of states expanded by the minimax algorithm, **with and without pruning**, to determine the minimax decision from the initial 3x3 Tic-Tac-Toe board state.  This can be done in many ways, but writing out all the states by hand is **not** one of them (as you will find out!).

Then compute the percent savings that you get by using alpha-beta pruning. i.e. Compute $\frac{\text{Number of nodes expanded with pruning}}{\text{Number of nodes expanded with minimax}} $

In [8]:
ttt = TicTacToe(3,3,3)
withPruning = alphabeta_player(ttt, prune = True, expandedTest = True)
withoutPruning = alphabeta_player(ttt, prune = False, expandedTest = True)

print('States expanded with pruning: ', withPruning) 
print('States expanded without pruning: ', withoutPruning)
print('Percent savings with pruning: ', 100 - (100*(withPruning/withoutPruning)), '%') 

States expanded with pruning:  30709
States expanded without pruning:  549945
Percent savings with pruning:  94.41598705325077 %
